In [ ]:
# Install the package (in Colab, use the path to your source)
!pip install -q peft transformers torch tqdm numpy

# If running locally from source:
import sys
import os
sys.path.append(os.path.abspath('../src')) 

import torch
from iladok import NoiseRouter

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Iladok installed. Running on {device.upper()}.")


In [ ]:
# 1. Connect the Local Core Engine
# We use GPT-2 Small for speed, but this works with other models too.
print("🔌 Connecting to Mother Model...")
router = NoiseRouter.from_pretrained("gpt2", device=device)
print("✅ Engine Online.")


In [ ]:
# 2. Define Steering Data (The "Mechanism")
# We separate knowledge into two distinct domains to prove the router works.

# Mechanism A: French Translation
french_data = [
    ("Hello", "Bonjour"),
    ("Thank you", "Merci"),
    ("My name is", "Je m'appelle"),
    ("Good morning", "Bon matin"),
    ("I love you", "Je t'aime"),
    ("Where is the library?", "Où est la bibliothèque?"),
    ("The wine is good", "Le vin est bon")
]

# Mechanism B: Spanish Translation
spanish_data = [
    ("Hello", "Hola"),
    ("Thank you", "Gracias"),
    ("My name is", "Me llamo"),
    ("Good morning", "Buenos días"),
    ("I love you", "Te quiero"),
    ("Where is the library?", "¿Dónde está la biblioteca?"),
    ("The weather is nice", "Hace buen tiempo")
]

print(f"Prepared {len(french_data)} French examples and {len(spanish_data)} Spanish examples.")


In [ ]:
# 3. Optimize Steering Vectors
# This usually takes ~2 mins on a GPU or longer on CPU for small models.

print("\n🚀 Optimizing 'French Mechanism' Vector...")
router.register_vector("french", data=french_data, epochs=5)

print("\n🚀 Optimizing 'Spanish Mechanism' Vector...")
router.register_vector("spanish", data=spanish_data, epochs=5)

print("\n✅ Optimization Complete. Vectors 'french' and 'spanish' are ready.")


In [ ]:
# 4. Run the Mechanism Check (Perplexity Routing)

test_inputs = [
    "The wine is good",      # Distinctly French concept
    "The weather is nice",   # Distinctly Spanish concept
    "Hello"                  # Common concept (The Trap!)
]

print(f"{'INPUT':<25} | {'WINNING MECHANISM':<20} | {'OUTPUT'}")
print("-" * 75)

for text in test_inputs:
    # This single line runs the 'auto_select' logic
    # It computes signal loss for both vectors and picks the winner
    output = router.generate(
        text, 
        steering_vectors=["french", "spanish"], 
        auto_select=True,
        max_new_tokens=10
    )
    
    # We can infer the winner from the output for this demo visualization
    winner = "FRENCH" if any(k in output for k in ["Bon", "vin", "Bonjour", "Bonjour"]) else "SPANISH"
    
    print(f"{text:<25} | {winner:<20} | {output.strip()}")
